In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
# 1. Load the finalized data
df = pd.read_csv('../data/processed/final.csv')

# 2. Split Features and Target
X = df.drop('target', axis=1)
y = df['target']

# 3. Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [3]:
# 4. Handle Class Imbalance (SMOTE)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 5. FIT THE SCALER (This is the critical step!)
# We fit only on training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)

# 6. Apply same scaler to test data
X_test_scaled = scaler.transform(X_test)

print("✅ Data scaled and balanced successfully.")


✅ Data scaled and balanced successfully.


In [4]:
# 7. Train the XGBoost Model on SCALED data
best_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
best_model.fit(X_train_scaled, y_train_res)

# 8. Evaluate
y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

# 9. SAVE THE ARTIFACTS (Overwrite the old ones)
joblib.dump(best_model, '../models/xgboost_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(list(X.columns), '../models/Feature_names.joblib')

print("🚀 Model, Scaler, and Feature Names saved successfully!")


              precision    recall  f1-score   support

           0       0.91      0.68      0.78      1857
           1       0.38      0.74      0.50       480

    accuracy                           0.70      2337
   macro avg       0.64      0.71      0.64      2337
weighted avg       0.80      0.70      0.72      2337

🚀 Model, Scaler, and Feature Names saved successfully!
